In [ ]:
from google.cloud import bigquery
import os
import plotly.express as px
import pandas as pd
from pyproj import Transformer

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../data/auth/sante-et-territoires-216daad01024.json" 
client = bigquery.Client() 

query = """
SELECT *
FROM `sante-et-territoires.finess.soin_etab`
"""

df = client.query(query).to_dataframe()
df_communes = pd.read_csv("../data/geo/communes-france-2025.csv", sep=",", encoding="utf-8")



In [ ]:
df.info()

In [ ]:

# Création du transformateur Lambert 93 -> WGS84
transformer = Transformer.from_crs("EPSG:2154", "EPSG:4326", always_xy=True)

# Conversion
df["longitude"], df["latitude"] = transformer.transform(
    df["coordxet"].values,df["coordyet"].values
)

# Aperçu
print(df[["nofinesset", "coordxet", "coordyet", "latitude", "longitude"]].head())

In [ ]:
df.head()

In [ ]:
df['libactivite'].value_counts()

In [ ]:
# Regroupement des catégories d'activités
def regrouper_categorie(cat):
    cat = cat.lower()

    if any(x in cat for x in ["greffe"]):
        return "Greffe"

    if any(x in cat for x in ["soins de suite"]):
        return "Soins de suite"

    if any(x in cat for x in ["chirurgie"]):
        return "Chirurgie"
    if any(x in cat for x in ["médecine d\'urgence"]):
        return "Médecine d'urgence"
    if any(x in cat for x in ["médecine"]):
        return "Médecine"
    if any(x in cat for x in ["médecine"]):
        return "Médecine"
    if any(x in cat for x in ["psychiatrie"]):
        return "Psychiatrie"
    if any(x in cat for x in ["amp dpn"]):
        return "AMP DPN"
    if any(x in cat for x in ["cancer"]):
        return "Cancer"
    if any(x in cat for x in ["gynécologie"]):
        return "Gynécologie"
    if any(x in cat for x in ["soins de longue durée"]):
        return "Soins de longue durée"
    if any(x in cat for x in ["insuffisance rénale"]):
        return "Insuffisance rénale"
    if any(x in cat for x in ["examen des caractéristiques génétiques"]):
        return "Examen des caractéristiques génétiques"
    return "Autres"

# Application
df["groupe_activites"] = df["libactivite"].apply(regrouper_categorie)

In [ ]:
df['groupe_activites'].value_counts()

In [ ]:
df.loc[df['groupe_activites'] == "Autres", 'libactivite']


In [ ]:
#nettoyage des doublons
df_clean = df.drop_duplicates()

In [ ]:
df_clean.info()

In [ ]:
#je crée un colonne code_insee à partir de la colonne code commune et de la colonne departement
df_clean.loc[:, "code_insee"] = (
    df_clean["departement"].astype(str).str.zfill(2)
    + df_clean["commune"].astype(str).str.zfill(3)
)

df_clean.head()

In [ ]:
# latitudes longitudes en float
df_clean.loc[:,'latitude'] = df_clean['latitude'].astype(float)
df_clean.loc[:,'longitude'] = df_clean['longitude'].astype(float)

In [ ]:
df_clean = df_clean.rename(columns={
    "rs": "raison_sociale",
    "libcategetab": "categorie",
    "commune": "code_commune",
    "nofinessej": "numero finess etablissement juridique",
      'nofinesset': "numero_finess_etablissement", 
        'libdepartement': "libelle_departement",
         'groupe' : "type d etablissements",
})

In [ ]:
#je filtre les données sur les départements de la région Occitanie
departements_occitanie = ["09","11", "12", "30","31", "32", "34", "46", "48","65", "66", "81", "82",  ]
df_occitanie = df_clean[df_clean["departement"].isin(departements_occitanie)]

In [ ]:
df_occitanie.head()

In [ ]:
#j'exporte en csv dans processed l occitanie plus les départements limitrophes
departement_limitrophes_et_occitanie= ["64", "40", "47", "24", "19", "15", "43", "07", "84", "13", "09","11", "12", "30","31", "32", "34", "46", "48","65", "66", "81", "82"]
df_limit= df_clean[df_clean["departement"].isin(departement_limitrophes_et_occitanie)]
df_limit.info()
df_limit.to_csv("../data/processed/soins_limit.csv", index=False, sep=";", encoding="utf-8")

In [ ]:
#j'exporte en csv dans processed la partie occitanie

df_occitanie.to_csv("../data/processed/soins_occitanie.csv", index=False, sep=";", encoding="utf-8")


In [ ]:
# Je crée un dataframe pour les urgences uniquement 
df_urgences_france = df_clean[df_clean['groupe_activites'].str.contains("urgence", case=False, na=False)]

Fonction Haversine (distance en km)
🌍 Principe général
La Terre est (approximativement) une sphère.
Pour calculer la distance entre deux points sur une sphère, on ne peut pas utiliser la distance euclidienne classique.
La formule de Haversine permet de calculer la distance orthodromique, c’est‑à‑dire la distance la plus courte entre deux points à la surface de la Terre.

In [ ]:
import numpy as np

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # rayon de la Terre en km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

In [ ]:
#filtre sur les communes de l'occitanie
df_communes_occitanie = df_communes[df_communes['reg_nom'] == 'Occitanie']

In [ ]:
# On crée une matrice de distances
distances = []

for idx, commune in df_communes_occitanie.iterrows():
    lat_c, lon_c = commune['latitude_centre'], commune['longitude_centre']
    
    # distances entre la commune et tous les établissements d'urgence
    d = haversine(
        lat_c, lon_c,
        df_urgences['latitude'].values,
        df_urgences['longitude'].values
    )
    
    # distance minimale = établissement le plus proche
    distances.append(d.min())

df_communes_occitanie['distance_urgence_km'] = distances


In [ ]:
#csv des distances des communes aux urgences en occitanie
df_communes_occitanie.to_csv("../data/processed/distances_communes_urgence_occitanie.csv", index=False)


In [ ]:
#filtre sur les communes de l'occitanie
df_communes_occitanie_tot_soin = df_communes[df_communes['reg_nom'] == 'Occitanie']

In [ ]:
# On crée une matrice de distances pour tous les soins
distances2 = []

for idx, commune in df_communes_occitanie.iterrows():
    lat_c, lon_c = commune['latitude_centre'], commune['longitude_centre']
    
    # distances entre la commune et tous les établissements d'urgence
    d = haversine(
        lat_c, lon_c,
        df_occitanie['latitude'].values,
        df_occitanie['longitude'].values
    )
    
    # distance minimale = établissement le plus proche
    distances2.append(d.min())

df_communes_occitanie_tot_soin['distance_urgence_km'] = distances2


In [ ]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, asin, sqrt
import plotly.express as px

# --- 1. Fonction haversine (en km) ---
def haversine(lat1, lon1, lat2, lon2):
    # lat1, lon1 : scalaires
    # lat2, lon2 : arrays numpy
    R = 6371  # rayon de la Terre en km
    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2, lon2 = np.radians(lat2), np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

# --- 2. Chargement des données ---


# --- 3. Filtrer les communes d'Occitanie ---
df_communes_occ = df_communes[df_communes["reg_nom"] == "Occitanie"].copy()

# On garde les colonnes utiles
df_communes_occ = df_communes_occ[[
    "code_insee", "nom_standard", "latitude_centre", "longitude_centre"
]].dropna()

# --- 4. Filtrer les établissements avec coordonnées et type de soin ---
df_soins = df.dropna(subset=["latitude", "longitude", "groupe_activites"]).copy()

# --- 5. Calcul des distances min par commune et par type de soin ---
resultats = []

types_soins = df_soins["groupe_activites"].unique()

for type_soin in types_soins:
    df_type = df_soins[df_soins["groupe_activites"] == type_soin]

    # coordonnées des établissements de ce type
    etab_lats = df_type["latitude"].values
    etab_lons = df_type["longitude"].values

    distances_min = []

    for idx, com in df_communes_occ.iterrows():
        lat_c = com["latitude_centre"]
        lon_c = com["longitude_centre"]

        d = haversine(lat_c, lon_c, etab_lats, etab_lons)
        distances_min.append(d.min())

    df_tmp = pd.DataFrame({
        "code_insee": df_communes_occ["code_insee"].values,
        "groupe_activites": type_soin,
        "distance_min_km": distances_min
    })

    resultats.append(df_tmp)

df_distances_types = pd.concat(resultats, ignore_index=True)

# --- 6. Moyenne des distances par type de soin ---
df_radar = (
    df_distances_types
    .groupby("groupe_activites", as_index=False)
    .agg(distance_moyenne_km=("distance_min_km", "mean"))
)

df_radar["distance_moyenne_km"] = df_radar["distance_moyenne_km"].round(2)

# --- 7. Radar Plotly ---
fig = px.line_polar(
    df_radar,
    r="distance_moyenne_km",
    theta="groupe_activites",
    line_close=True,
    title="Distance moyenne des communes d'Occitanie aux différents types de soins"
)
fig.update_traces(fill="toself")

fig.show()


In [ ]:
import logging
import os
import plotly.express as px
import pandas as pd
from pyproj import Transformer

from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent  # dossier du .py
df_soins = pd.read_csv(BASE_DIR / "../data/raw/finess_activites_soin.csv", sep=";", encoding="utf-8")
df_communes = pd.read_csv(BASE_DIR / "../data/geo/communes-france-2025.csv", sep=",", encoding="utf-8")




# ---------------------------------------------------------------------------
# Fonctions de nettoyage
# ---------------------------------------------------------------------------

def clean_nofiness(series: pd.Series) -> pd.Series:
    """
    Normalise les numéros FINESS : chaîne de 9 caractères, zéros à gauche.
    Exemples : 10001246 → '010001246', 310001234 → '310001234'
    """
    return series.astype(str).str.zfill(9)


def clean_department(code: str) -> str:
    """Normalise un code département : '9' → '09', '31' → '31'."""
    if pd.isna(code):
        return None
    code = str(code).strip().zfill(2)
    return code[:2]  # Garde uniquement les 2 premiers chiffres


def lambert93_to_wgs84(x: float, y: float):
    """
    Convertit des coordonnées Lambert93 (EPSG:2154) en WGS84 (lat/lon).
    Implémentation pure Python basée sur les paramètres IAG GRS80.
    """
    try:
        if pd.isna(x) or pd.isna(y) or float(x) == 0 or float(y) == 0:
            return None, None
        x, y = float(x), float(y)

        # Paramètres ellipsoïde GRS80
        a = 6378137.0
        e = 0.0818191910428158

        # Paramètres projection Lambert93
        lc = math.radians(3.0)          # Longitude centrale
        phi0 = math.radians(46.5)       # Latitude d'origine
        phi1 = math.radians(44.0)       # Parallèle 1
        phi2 = math.radians(49.0)       # Parallèle 2
        x0 = 700000.0
        y0 = 6600000.0

        def _geo_lat(phi):
            sp = e * math.sin(phi)
            return math.tan(math.pi / 4 + phi / 2) * ((1 - sp) / (1 + sp)) ** (e / 2)

        m1 = math.cos(phi1) / math.sqrt(1 - (e * math.sin(phi1)) ** 2)
        m2 = math.cos(phi2) / math.sqrt(1 - (e * math.sin(phi2)) ** 2)
        t1 = _geo_lat(phi1)
        t2 = _geo_lat(phi2)
        t0 = _geo_lat(phi0)

        n = math.log(m1 / m2) / math.log(t1 / t2)
        F = m1 / (n * t1 ** n)
        r0 = a * F * t0 ** n

        dx, dy = x - x0, y - y0 + r0
        r = math.sqrt(dx ** 2 + dy ** 2) * math.copysign(1, n)
        theta = math.atan(dx / (r0 - (y - y0)))
        lon = math.degrees(theta / n + lc)

        t = (r / (a * F)) ** (1 / n)
        phi = math.pi / 2 - 2 * math.atan(t)
        for _ in range(10):
            sp = e * math.sin(phi)
            phi = math.pi / 2 - 2 * math.atan(t * ((1 - sp) / (1 + sp)) ** (e / 2))

        lat = math.degrees(phi)

        if 41 <= lat <= 52 and -5 <= lon <= 10:
            return round(lat, 6), round(lon, 6)
        return None, None
    except Exception:
        return None, None

   
#nettoyage activité soins
# Sélection des colonnes utiles
cols = ["nofinesset", "nofinessej", "libactivite", "libmodalite", "libforme", "datemeo", "datefin"]
df_soins = df[[c for c in cols if c in df.columns]].copy()
#nettoyage nofinesset
df_soins["nofinesset"] = clean_nofiness(df["nofinesset"])
df_soins["nofinessej"] = clean_nofiness(df["nofinessej"])

# Regroupement des catégories d'activités pour les soins
def regrouper_categorie(cat):
    cat = cat.lower()

    if any(x in cat for x in ["greffe"]):
        return "Greffe"

    if any(x in cat for x in ["soins de suite"]):
        return "Soins de suite"

    if any(x in cat for x in ["chirurgie"]):
        return "Chirurgie"
    if any(x in cat for x in ["médecine d\'urgence"]):
        return "Médecine d'urgence"
    if any(x in cat for x in ["médecine"]):
        return "Médecine"
    if any(x in cat for x in ["médecine"]):
        return "Médecine"
    if any(x in cat for x in ["psychiatrie"]):
        return "Psychiatrie"
    if any(x in cat for x in ["amp dpn"]):
        return "AMP DPN"
    if any(x in cat for x in ["cancer"]):
        return "Cancer"
    if any(x in cat for x in ["gynécologie"]):
        return "Gynécologie"
    if any(x in cat for x in ["soins de longue durée"]):
        return "Soins de longue durée"
    if any(x in cat for x in ["insuffisance rénale"]):
        return "Insuffisance rénale"
    if any(x in cat for x in ["examen des caractéristiques génétiques"]):
        return "Examen des caractéristiques génétiques"
    return "Autres"

# Application
df_soins["groupe_activites"] = df_soins["activite"].apply(regrouper_categorie)




